# Week 4 Exercise — Stayez Python-to-C++ Algorithm Optimizer

**Student:** Vagz1216  
**Course:** LLM Engineering — Andela AI Engineering Bootcamp  
**Exercise:** Week 4 — AI-Assisted Code Engineering (Python to C++)

---

## Project Overview
For Week 4, I am applying the **AI code engineering** concept to the Stayez platform context. Stayez properties currently have static pricing in WooCommerce. However, as the platform scales, we want to introduce **Dynamic Surge Pricing** (based on weekends, local competition, and popularity).

Calculating this for 100,000 concurrent searches in Python would crash the server. This tool acts as an AI-powered **Senior C++ Engineer** — it takes the slow Python logic for our new pricing engine and translates it into blazingly fast, optimized C++ code that handles millions of calculations per second.

### Features:
- Uses **Hugging Face (Llama 3.2 3B)**, **Groq (Llama 3.3 70B)**, and **Gemini 1.5 Flash** as code generation engines.
- Side-by-side Gradio UI with syntax-highlighted Python and C++ code panels.
- Pre-loaded with a highly realistic Stayez dynamic pricing simulation algorithm.


In [ ]:
# Cell 1: Imports
import os
import io
import sys
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
from huggingface_hub import InferenceClient

In [ ]:
# Cell 2: API Keys and Clients Setup
load_dotenv(override=True)

groq_api_key   = os.getenv('GROQ_API_KEY')
gemini_api_key = os.getenv('GEMINI_API_KEY')
hf_token       = os.getenv('HF_TOKEN')

# 1. Hugging Face Inference Client (Open Source via HF API)
hf_client = InferenceClient(token=hf_token)

# 2. Groq Client (Fastest inference for 70B open-source models)
groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")

# 3. Gemini Client (Proprietary frontier model fallback)
gemini = OpenAI(api_key=gemini_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

# Model registry: name -> (model_id, client)
MODELS = {
    "Groq (Llama 3.3 70B)": ("llama-3.3-70b-versatile", groq),
    "Hugging Face (Llama 3.2 3B)": ("meta-llama/Llama-3.2-3B-Instruct", hf_client),
    "Gemini 1.5 Flash": ("gemini-1.5-flash", gemini),
}

print("✅ Clients ready: Hugging Face, Groq, Gemini")

In [ ]:
# Cell 3: System Prompt

system_prompt = """\
You are a Senior C++ Software Engineer working on performance-critical backend systems.
Your task is to translate Python code into high-performance, idiomatic C++ code.

STRICT RULES:
1. Respond ONLY with raw C++ code. No markdown code fences, no explanations.
2. Include necessary #include headers (<iostream>, <cmath>, <iomanip>, etc) and a main() function.
3. Use appropriate data types like double for precision and int64_t for large loops.
4. Prioritize maximum runtime performance.
5. Produce identical output to the original Python code, formatting numbers consistently.
"""

def build_user_prompt(python_code):
    return f"Convert this Python code to high-performance C++:\n\n{python_code}"

In [ ]:
# Cell 4: Port Function

def port_to_cpp(model_choice, python_code):
    if not python_code.strip():
        return "// Please paste some Python code to translate."

    model_id, client = MODELS[model_choice]
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": build_user_prompt(python_code)}
    ]

    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=messages,
            temperature=0.1,
            max_tokens=4000
        )
        reply = response.choices[0].message.content.strip()
        # Strip any markdown fences the model may add
        for fence in ["```cpp", "```c++", "```c", "```"]:
            reply = reply.replace(fence, "")
        return reply.strip()

    except Exception as e:
        return f"// Translation error:\n// {e}"


def run_python(python_code):
    """Execute the Python code and capture its stdout output."""
    buf = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buf
    try:
        exec(python_code, {"__builtins__": __builtins__})
        return buf.getvalue()
    except Exception as e:
        return f"Error: {e}"
    finally:
        sys.stdout = old_stdout

In [ ]:
# Cell 5: Realistic Stayez Benchmark Python Code

python_example = """# Stayez Dynamic Pricing Benchmark 
# Simulates pricing calculations for 1,000,000 concurrent guest configurations
import time
import math

def calculate_stayez_price(base_ksh, check_in_days_from_now, nights, 
                           competing_properties, reviews_count):
    total = 0.0
    for night in range(nights):
        day_price = base_ksh
        current_day = check_in_days_from_now + night
        
        # Weekend surge: +30% (Friday/Saturday/Sunday)
        if current_day % 7 >= 4:
            day_price *= 1.30
            
        # Scarcity factor: fewer competitors = charge more
        scarcity = math.exp(-competing_properties / 10.0)
        day_price *= (1 + 0.2 * scarcity)
        
        # Popularity premium: more reviews = trusted = priced higher 
        popularity_premium = math.log1p(reviews_count) * 0.01
        day_price *= (1 + popularity_premium)
        
        total += day_price
        
    return total

def run_stayez_pricing_benchmark(iterations):
    grand_total_revenue = 0.0
    
    # Simulate calculating prices for a million different booking scenarios
    for i in range(iterations):
        days_from_now = i % 30
        nights = (i % 5) + 1
        competing = (i % 20)
        reviews = i % 100
        
        price = calculate_stayez_price(5000, days_from_now, nights, competing, reviews)
        grand_total_revenue += price
        
    return round(grand_total_revenue, 2)

start = time.time()
total_projected_revenue = run_stayez_pricing_benchmark(1_000_000)
end = time.time()

print(f\"Total Projected Revenue: KSh {total_projected_revenue:,.2f}\")
print(f\"Calculation took {end - start:.4f} seconds in Python\")
"""

In [ ]:
# Cell 6: Gradio UI

with gr.Blocks(theme=gr.themes.Soft(), title="Stayez Python-to-C++ Optimizer") as demo:
    gr.Markdown("## Stayez Algorithm Optimizer — Python to C++")
    gr.Markdown(
        "Use frontier AI models to translate slow Python pricing algorithms into high-performance C++ code.  \n"
        "Select a model, then click **Convert to C++**."
    )

    with gr.Row():
        model_dd = gr.Dropdown(
            choices=list(MODELS.keys()),
            value="Groq (Llama 3.3 70B)",
            label="AI Code Engineer",
            scale=3
        )
        convert_btn = gr.Button("Convert to C++ →", variant="primary", scale=1)

    with gr.Row():
        with gr.Column():
            python_input = gr.Code(
                label="Python Code (Source)",
                value=python_example,
                language="python",
                lines=35
            )
            run_py_btn = gr.Button("▶ Run Python")
            python_out = gr.Textbox(label="Python Output", lines=3, interactive=False)

        with gr.Column():
            cpp_output = gr.Code(
                label="C++ Code (Generated)",
                value="// Click 'Convert to C++' to generate...",
                language="cpp",
                lines=35
            )

    convert_btn.click(fn=port_to_cpp, inputs=[model_dd, python_input], outputs=[cpp_output])
    run_py_btn.click(fn=run_python, inputs=[python_input], outputs=[python_out])

demo.launch(inbrowser=True)